Clustering usando o K-means clustering para o Objetivo 1:


Identificar e agrupar perfis de clientes para perceber em que grupos o exercício físico ou o contacto social têm maior efeito.

In [ ]:
# clustering_exercicio_social.py

import sqlite3
import pandas as pd
from datetime import datetime

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# Caminho e tabela
db_path = r"C:\BaseDadosTeste\ScreenTimevsMentalWellness.db"
main_table = "ScreenTimevsMentalWellness"

# 1) Ler dados
conn = sqlite3.connect(db_path)
df = pd.read_sql_query(f"SELECT * FROM {main_table}", conn)
conn.close()

print("Dimensão original:", df.shape)

# 2) Colunas para o objetivo A (exercício, social, bem-estar)
feature_cols = [
    "exercise_minutes_per_week",
    "social_hours_per_week",
    "sleep_hours",
    "sleep_quality_1_5",
    "stress_level_0_10",
    "productivity_0_10",
    "mental_wellness_index_0_10",
]

feature_cols = [c for c in feature_cols if c in df.columns]
print("Colunas usadas no clustering (objetivo A):", feature_cols)

X = df[feature_cols].copy()
X = X.fillna(X.mean())

# 3) Normalizar
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 4) Método do cotovelo para escolher k
inertias = []
ks = range(2, 11)

for k in ks:
    km_tmp = KMeans(n_clusters=k, random_state=42, n_init=10)
    km_tmp.fit(X_scaled)
    inertias.append(km_tmp.inertia_)

plt.figure()
plt.plot(ks, inertias, marker="o")
plt.xlabel("Número de clusters (k)")
plt.ylabel("Inércia")
plt.title("Objetivo A - Método do cotovelo")
plt.grid(True)
plt.show()

# 5) Escolher k final depois de ver o gráfico
k_final = 4  # ajusta este valor conforme o cotovelo

kmeans = KMeans(n_clusters=k_final, random_state=42, n_init=10)
df["cluster_exercicio_social"] = kmeans.fit_predict(X_scaled)

print("Número de pontos por cluster (objetivo A):")
print(df["cluster_exercicio_social"].value_counts().sort_index())

# 6) Perfis médios por cluster
summary_cols = [
    "exercise_minutes_per_week",
    "social_hours_per_week",
    "sleep_hours",
    "sleep_quality_1_5",
    "stress_level_0_10",
    "productivity_0_10",
    "mental_wellness_index_0_10",
]
summary_cols = [c for c in summary_cols if c in df.columns]

cluster_profile = df.groupby("cluster_exercicio_social")[summary_cols].mean()
print("\nPerfis médios por cluster (objetivo A):")
print(cluster_profile)

# 7) PCA para visualização 2D
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

df["pca1_A"] = X_pca[:, 0]
df["pca2_A"] = X_pca[:, 1]

plt.figure(figsize=(8, 6))
scatter = plt.scatter(
    df["pca1_A"],
    df["pca2_A"],
    c=df["cluster_exercicio_social"],
    cmap="tab10",
    alpha=0.8
)
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.title("Objetivo A - Clusters em 2D (PCA)")
plt.grid(True)
plt.show()

# 8) Guardar na BD com backup
conn = sqlite3.connect(db_path)

backup_name = f"Backup_ObjA_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
df.to_sql(backup_name, conn, if_exists="replace", index=False)
print(f"Backup com clusters do objetivo A criado: {backup_name}")

df.to_sql(main_table, conn, if_exists="replace", index=False)
conn.close()

print("Tabela principal atualizada com a coluna 'cluster_exercicio_social' e coordenadas PCA (pca1_A, pca2_A).")
